# Clase 19 — Docker Network y Docker Compose

**Objetivo:** conectar la API y Streamlit que ya construiste, mediante Docker Network y Docker Compose.

Desde la raíz del repositorio público, revisa `git status` y actualiza `main` con `git pull`; revisa cualquier cambio rastreado antes de actualizar. Ejecuta `uv sync --locked`.

**Punto de partida:** continúa en `labs/trabajo-local/clase-16/`, con `api/`, `ui/` y sus archivos existentes. Consulta el [notebook Docker](clase-17-docker.ipynb) para el repaso de imágenes y el [contrato de API y Streamlit](../modulo-02-ciclo-mlops/clase-14-model-service-streamlit.ipynb).

**Resultado observable:** una predicción desde Streamlit hacia API, primero con `--network host` y después con el montaje descrito en `docker-compose.yaml`.

Usa Git Bash en Windows, Zsh en macOS o Bash en Linux, con contenedores Linux en Docker Desktop o Docker Engine en Linux. El trabajo permanece local e ignorado por Git, sin entrega, commit ni push.

## 1. Contexto actual: una API y una interfaz

La carpeta de trabajo contiene los archivos que ya utilizaste:

```text
labs/trabajo-local/clase-16/
├── docker-compose.yaml
├── api/
│   ├── api.py
│   ├── Dockerfile
│   ├── .dockerignore
│   ├── requirements.txt
│   └── .env
└── ui/
    ├── streamlit_app.py
    ├── Dockerfile
    ├── .dockerignore
    ├── requirements.txt
    └── .env
```

La API carga el modelo y expone `POST /api/v1/predicciones`. Streamlit captura las cinco features y hace la petición HTTP desde **su proceso Python**; el navegador sólo muestra la interfaz.

```text
Navegador → Streamlit → HTTP → API → Modelo
            contenedor UI    contenedor API
```

Desde la raíz del repositorio, entra al trabajo existente:

```bash
cd labs/trabajo-local/clase-16
pwd
ls -a
ls -a api
ls -a ui
docker ps -a
docker images nyc-taxi-api-image
docker images nyc-taxi-ui-image
```

`pwd` confirma la ubicación y `ls -a` muestra también archivos ocultos. No imprimas los valores de `.env` al compartir pantalla.

### Nombres de imágenes, contenedores y servicios

| Elemento | API | UI |
|---|---|---|
| Imagen y tag | `nyc-taxi-api-image:v1` | `nyc-taxi-ui-image:v1` |
| Contenedor de esta práctica | `nyc-taxi-api-container` | `nyc-taxi-ui-container` |
| Servicio de Compose | `api` | `ui` |
| Puerto interno | 8000 | 8501 |

La imagen es el paquete construido; el contenedor es una instancia y el servicio es su definición en Compose. `v1` es el tag de la imagen, no la versión del modelo.

Si Compose no declara `container_name`, genera nombres como `clase-16-api-1` y `clase-16-ui-1`. Son nombres de instancias asignados por Compose. La propiedad `container_name` permite fijar los nombres de la tabla.

**Checkpoint:** estás dentro de `clase-16/` y reconoces tus archivos, imágenes y contenedores existentes.

## 2. Revisar los Dockerfiles y la configuración existente

No es necesario reescribir la API ni la interfaz. Compara los Dockerfiles que ya creaste:

| Instrucción | API | UI |
|---|---|---|
| `FROM` | `python:3.12-slim` | `python:3.12-slim` |
| `WORKDIR` | `/code` | `/code` |
| Dependencias | Copiar `requirements.txt` e instalarlo | El mismo procedimiento con los requisitos de UI |
| Código | `COPY . /code/` desde `api/` | `COPY . /code/` desde `ui/` |
| `EXPOSE` | 8000 | 8501 |
| `CMD` | Uvicorn con `api:app` | Streamlit con `streamlit_app.py` |

`WORKDIR` fija el directorio interno. `COPY` lee dentro del contexto; `EXPOSE` documenta el puerto y `CMD` define el proceso de arranque. Ambos servidores deben escuchar en `0.0.0.0` para recibir conexiones por la interfaz del contenedor.

### Excluir la configuración del build

En **ambos** archivos, `api/.dockerignore` y `ui/.dockerignore`, comprueba estas líneas:

```text
.env
.env.*
.venv/
.git/
__pycache__/
*.pyc
*.ipynb
```

`.env.*` excluye nombres como `.env.local`, pero **no excluye `.env`**. Una exclusión vacía tampoco evita que `COPY .` incorpore configuración personal. Cada contexto necesita su propio `.dockerignore`. Los `.env` permanecen en sus carpetas del host y se entregan al ejecutar.

**Checkpoint:** los Dockerfiles corresponden a sus componentes y ambos contextos excluyen `.env`. Conserva los scripts y requisitos existentes.

## 3. Repaso: construir imágenes y ejecutar contenedores

**Los comandos de esta práctica se ejecutan desde `labs/trabajo-local/clase-16/`.** En cada terminal que abras en la raíz del repositorio, entra primero a esa carpeta:

```bash
cd labs/trabajo-local/clase-16
pwd
ls -a
```

Debes ver `api/`, `ui/` y `docker-compose.yaml`. Conserva los requisitos de cada componente.

### Construir las imágenes

```bash
docker build -t nyc-taxi-api-image:v1 ./api
docker build -t nyc-taxi-ui-image:v1 ./ui
```

`-t` asigna nombre y tag; `./api` y `./ui` seleccionan los contextos. Docker busca el archivo `Dockerfile` en cada contexto. Reconstruir una imagen no cambia contenedores que ya estén activos.

Antes de ejecutar, revisa `docker ps -a`: los nombres y puertos de la práctica deben estar libres. Si tus contenedores previos se llaman `clase-16-ui-1` y `clase-16-api-1`, detén sólo esas instancias con `docker stop clase-16-ui-1 clase-16-api-1`. Si tienen otros nombres, identifica cuáles corresponden a tu trabajo antes de detenerlos.

### Interpretar `-p 8000:8000`

La forma básica es **`PUERTO_HOST:PUERTO_CONTENEDOR`**. El número izquierdo corresponde a tu computadora y el derecho al proceso dentro del contenedor.

| Publicación | Navegador en tu computadora | API dentro del contenedor |
|---|---|---|
| `8000:8000` | Puerto 8000 | Puerto 8000 |
| `8001:8000` | Puerto 8001 | Puerto 8000 |

Los números no tienen que coincidir: con `8001:8000`, abres `http://127.0.0.1:8001/health`, mientras Uvicorn conserva 8000. El prefijo `127.0.0.1` en nuestros comandos limita la publicación a la propia computadora.

### Ejecutar por separado

Para reproducir el problema de conexión, deja inicialmente en `ui/.env`:

```text
API_URL=http://127.0.0.1:8000/api/v1/predicciones
```

Conserva en `api/.env` la configuración del modelo que ya funciona.

**Terminal 1 — API**, desde `clase-16/`:

```bash
docker run --rm --name nyc-taxi-api-container --env-file ./api/.env -p 127.0.0.1:8000:8000 nyc-taxi-api-image:v1
```

**Terminal 2 — Streamlit**, también desde `clase-16/`:

```bash
docker run --rm --name nyc-taxi-ui-container --env-file ./ui/.env -p 127.0.0.1:8501:8501 nyc-taxi-ui-image:v1
```

Espera a que API termine de cargar el modelo. Abre `http://127.0.0.1:8000/health` y `http://127.0.0.1:8501`. Pulsa **Estimar duración**.

**Resultado esperado:** los dos servicios abren desde el navegador, pero la UI no logra pedir la predicción. Si `/health` tampoco responde, revisa primero el arranque de API y la carga del modelo.

## 4. ¿Qué ocurre al pulsar Estimar duración?

En la ejecución separada que acabas de probar, Streamlit intenta llamar a `http://127.0.0.1:8000/api/v1/predicciones`. Dentro de ese contenedor aislado, `127.0.0.1` y `localhost` apuntan a **la propia UI**, no a API.

### Cada contenedor es una casa

Decir «mi casa» depende de quién habla. Para llegar a otra casa necesitas su dirección dentro de un vecindario compartido. Publicar los dos puertos no convierte el localhost de UI en el localhost de API.

| Quién hace la petición | Qué significa `127.0.0.1:8000` |
|---|---|
| Navegador en tu computadora | La API accesible por el puerto publicado. |
| Python dentro de UI | El puerto 8000 del propio contenedor UI. |

La llamada a API sale del proceso Python de Streamlit. Para comunicar los dos componentes necesitamos una red compartida y una URL que identifique a API dentro de esa red.

## 5. Network host: compartir la red del anfitrión

El modo **host** permite que los contenedores utilicen la red del anfitrión. API y Streamlit siguen siendo procesos separados, con sus propios archivos y dependencias, pero comparten el entorno de red.

Por eso la UI puede utilizar **`http://127.0.0.1:8000/api/v1/predicciones`** para llamar a API. Esa dirección se conserva al ejecutar Python localmente, al iniciar los contenedores con host y al usar Compose con host.

```text
Red del anfitrión
Navegador → Streamlit (8501) → API (8000) → Modelo
                            127.0.0.1:8000
```

### ¿Qué simplifica?

- Se conserva la URL local de API en `ui/.env`.
- Los contenedores se ejecutan con `--network host`, sin crear otra red.
- No se publica un puerto mediante `-p`; la aplicación utiliza directamente su puerto de escucha.
- Compose expresa la misma decisión mediante `network_mode: host` en cada servicio.

El puerto 8000 debe estar libre para API y 8501 para Streamlit. **Con host, Docker descarta `-p`**: no puedes resolver un puerto ocupado remapeando `8001:8000`. Identifica el proceso propio que lo ocupa y deténlo, o cambia el puerto de escucha de la aplicación y su URL.

### Habilitar host networking

Detén API y UI con `Ctrl+C` en sus terminales antes de cambiar el modo de red o reiniciar Docker Desktop.

| Plataforma | Requisito |
|---|---|
| Docker Engine en Linux | Host está disponible de forma nativa. |
| Docker Desktop en Windows o macOS | Versión 4.34 o posterior, contenedores Linux y la opción Enable host networking habilitada. |

En Docker Desktop:

1. Inicia sesión y abre **Settings → Resources → Network**.
2. Activa **Enable host networking**.
3. Aplica con **Apply & restart**. Guarda el trabajo antes del reinicio.

Si la opción no aparece o está bloqueada, revisa versión y configuración con el profesor antes de continuar. No cambies al azar opciones de red o aislamiento.

Host comparte la red del anfitrión; **no aporta aislamiento de red entre el contenedor y el host**. El alcance del servicio depende también de la dirección de escucha y del firewall. Omitir `-p` no significa que el servicio esté limitado a loopback.

Referencia: [modo host, plataformas y configuración de Docker Desktop](https://docs.docker.com/engine/network/drivers/host/).

## 6. Ejecutar API y UI con network host

Con las instancias anteriores detenidas, continúa desde **`labs/trabajo-local/clase-16/`**.

### Conservar la URL en ui/.env

```text
API_URL=http://127.0.0.1:8000/api/v1/predicciones
```

El archivo mantiene la misma dirección que usaste localmente. `--env-file` entrega su valor al contenedor; `--network host` permite que esa dirección llegue a API. La variable configura el destino y el modo de red determina cómo se conecta el proceso.

En `ui/streamlit_app.py`, deja la lectura de configuración así:

```python
load_dotenv(override=False)

API_URL = os.getenv(
    "API_URL",
    "http://127.0.0.1:8000/api/v1/predicciones",
)
```

`os.getenv` lee la variable del proceso; el segundo argumento es el valor predeterminado si no existe. `override=False` evita reemplazar una variable que Docker ya entregó. El `.env` permanece fuera de la imagen gracias a `.dockerignore`. Conserva `api/.env` con la configuración del modelo.

Si ajustaste el código de la UI, incorpora el cambio a su imagen:

```bash
docker build -t nyc-taxi-ui-image:v1 ./ui
```

### Iniciar ambos contenedores

**Terminal 1 — API**, desde `clase-16/`:

```bash
docker run --rm --network host --name nyc-taxi-api-container --env-file ./api/.env nyc-taxi-api-image:v1
```

**Terminal 2 — Streamlit**, también desde `clase-16/`:

```bash
docker run --rm --network host --name nyc-taxi-ui-container --env-file ./ui/.env nyc-taxi-ui-image:v1
```

Ambos utilizan la red host y conservan los nombres de imagen y contenedor del repaso. Espera a que API termine de cargar el modelo. Comprueba `http://127.0.0.1:8000/health` y solicita la predicción en `http://127.0.0.1:8501`.

**Checkpoint:** Streamlit muestra minutos y metadatos del modelo utilizando la URL local de API.

Si cambias un `.env`, detén y vuelve a crear su contenedor para que reciba el valor nuevo. Ese cambio de configuración externa no exige reconstruir; un cambio del código copiado por Dockerfile sí lo exige.

## 7. Docker Compose: definir la aplicación en un archivo

Con comandos separados hay que conservar imágenes, contenedores, red, puertos, variables y comandos de arranque. **Docker Compose** describe esa aplicación en un archivo YAML y permite construirla y ejecutarla con un comando.

Un **servicio** es la definición de un componente: `api` o `ui`. Cada servicio utiliza una imagen y se ejecuta como un contenedor. El Dockerfile describe cómo construir cada imagen; Compose describe cómo ejecutar y relacionar los componentes.

### Preparar el archivo existente

Detén API y UI con `Ctrl+C` en sus terminales. Como se crearon con `--rm`, se retiran al terminar. El montaje host no creó una red propia que debas retirar.

Completa el **`docker-compose.yaml` que ya existe en `clase-16/`**. Conserva las carpetas `api/` y `ui/`, los Dockerfiles y sus `.env`.

### Plantilla completa con network host

La ejecución requiere host networking habilitado y conserva en `ui/.env`:

```text
API_URL=http://127.0.0.1:8000/api/v1/predicciones
```

La plantilla mantiene imágenes, nombres, contextos, comandos y archivos de configuración. La red se selecciona con `network_mode: host`; **no se incluyen `ports`, `networks` ni una URL distinta en `environment`**.

```yaml
name: clase-16

services:
  ui:
    image: nyc-taxi-ui-image:v1
    container_name: nyc-taxi-ui-container
    build:
      context: ./ui
      dockerfile: Dockerfile
    env_file:
      - ./ui/.env
    network_mode: host
    command: ["streamlit", "run", "streamlit_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
    depends_on:
      - api

  api:
    image: nyc-taxi-api-image:v1
    container_name: nyc-taxi-api-container
    build:
      context: ./api
      dockerfile: Dockerfile
    env_file:
      - ./api/.env
    network_mode: host
    command: ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]

```

YAML agrupa propiedades mediante sangría de espacios. `services` contiene `ui` y `api`; las propiedades indentadas pertenecen al servicio correspondiente. `network_mode: host` aplica a cada servicio; no se declara una red adicional al final del archivo.

Conserva un solo archivo de configuración. Los comandos usarán `-f docker-compose.yaml` para seleccionarlo expresamente.

## 8. Interpretar la plantilla de Compose

### Proyecto, imagen, contenedor y servicio

| Propiedad | Significado en el ejemplo |
|---|---|
| `name: clase-16` | Identifica el proyecto Compose. |
| `image` | Nombre y tag de cada imagen: `nyc-taxi-api-image:v1` y `nyc-taxi-ui-image:v1`. |
| `container_name` | Conserva nuestros nombres `nyc-taxi-api-container` y `nyc-taxi-ui-container`. |
| `services` | Define los componentes con los nombres `api` y `ui`. |

Sin `container_name`, Compose genera nombres como `clase-16-api-1`. Los nombres fijos son adecuados para esta práctica con una instancia por componente.

### Build: contexto y nombre del Dockerfile

`context: ./api` selecciona la carpeta cuyos archivos puede leer el build. Se interpreta desde la carpeta de `docker-compose.yaml`. `dockerfile: Dockerfile` selecciona el archivo de instrucciones **relativo a ese contexto**.

El archivo no tiene que llamarse siempre `Dockerfile`. Si dentro de `api/` se llamara `Dockerfile.api`, la configuración sería:

```yaml
build:
  context: ./api
  dockerfile: Dockerfile.api
```

| Archivo existente | Contexto | Valor de `dockerfile` |
|---|---|---|
| `api/Dockerfile` | `./api` | `Dockerfile` |
| `api/Dockerfile.api` | `./api` | `Dockerfile.api` |
| `ui/Dockerfile.ui` | `./ui` | `Dockerfile.ui` |

Son variantes de nombre; **mantén `Dockerfile` en tus carpetas actuales**. Renombrarlo no cambia el contexto: `COPY` continúa leyendo desde `api/` o `ui/`, y sus destinos siguen perteneciendo a la imagen. `WORKDIR /code` sigue siendo una ruta interna del contenedor.

### ¿Por qué aquí no hay ports?

En el repaso, `-p 127.0.0.1:8000:8000` publicaba el puerto del contenedor en tu computadora. En Compose, esa publicación se expresaría con `ports`. `EXPOSE 8000` sólo documenta el puerto y no crea una publicación.

**El modo host utiliza directamente los puertos de escucha de los procesos**, por lo que esta plantilla no incluye `ports`. Tampoco se utiliza esa propiedad para restringir el servicio a loopback o para remapear puertos. La API conserva 8000 y Streamlit 8501 porque sus comandos de arranque los seleccionan.

### ¿Por qué command si el Dockerfile ya tiene CMD?

El `CMD` del Dockerfile es el comando predeterminado de la imagen. `command` en Compose **lo reemplaza para ese servicio**. Si `command` se omite, se utiliza el `CMD` de la imagen.

En la plantilla, ambos coinciden para que el archivo muestre explícitamente cómo arranca cada componente. No se ejecutan dos procesos: sólo el comando elegido para esa instancia. La imagen puede reutilizarse con otro comando de ejecución sin modificar su Dockerfile.

### Variables y dirección de API

`env_file` entrega `api/.env` a API y `ui/.env` a UI. Las rutas se interpretan desde la carpeta del archivo Compose, no desde `/code`.

Con host, UI conserva en `ui/.env` la URL `http://127.0.0.1:8000/api/v1/predicciones`, tanto en Python local como en contenedores. El código lee la misma variable con `os.getenv`; `load_dotenv(override=False)` respeta los valores recibidos. No necesitamos `environment` para cambiar la URL en esta plantilla.

### network_mode: host

Cada servicio declara `network_mode: host`, equivalente a `--network host` en los comandos manuales. Compose no crea una red propia para estos servicios. No agregues `ports` ni referencias `networks` a esta plantilla.

### Orden de arranque

`depends_on: - api` establece que API se inicia antes que UI. **No espera a que el modelo termine de cargar.** Antes de solicitar la predicción, comprueba en los logs que API ya está atendiendo peticiones. Si haces clic demasiado pronto, espera y vuelve a intentar.

Referencias: [build y Dockerfile](https://docs.docker.com/reference/compose-file/build/), [puertos y command](https://docs.docker.com/reference/compose-file/services/), [variables](https://docs.docker.com/compose/how-tos/environment-variables/set-environment-variables/) y [modo host](https://docs.docker.com/engine/network/drivers/host/).

## 9. Construir y ejecutar con Docker Compose

Desde **`labs/trabajo-local/clase-16/`**:

```bash
docker compose -f docker-compose.yaml up --build
```

`-f` selecciona el archivo; **`up --build` construye las imágenes y después crea o actualiza los contenedores y aplica la configuración de red**. API y UI muestran sus logs en la terminal. Las capas que no cambiaron pueden reutilizar caché: `--build` no significa reconstruir todo sin caché.

Espera a que API termine de cargar el modelo. Comprueba `http://127.0.0.1:8000/health` y solicita una predicción desde `http://127.0.0.1:8501`.

**Checkpoint:** la misma aplicación predice con Compose, sin repetir los comandos manuales de ejecución. Se conserva la URL local de API.

### Los demás comandos básicos

Utiliza otra terminal ubicada también en `clase-16/` para consultar el montaje:

| Acción | Comando |
|---|---|
| Validar YAML sin imprimir valores de configuración | `docker compose -f docker-compose.yaml config --quiet` |
| Ver estado de los servicios | `docker compose -f docker-compose.yaml ps` |
| Ver los logs de ambos servicios | `docker compose -f docker-compose.yaml logs` |
| Ver sólo los logs de API | `docker compose -f docker-compose.yaml logs api` |
| Construir imágenes sin ejecutar | `docker compose -f docker-compose.yaml build` |
| Reconstruir imágenes sin reutilizar capas | `docker compose -f docker-compose.yaml build --no-cache` |
| Ejecutar sin solicitar una nueva construcción | `docker compose -f docker-compose.yaml up` |
| Detener sin retirar los contenedores | `docker compose -f docker-compose.yaml stop` |
| Detener y retirar los recursos del proyecto | `docker compose -f docker-compose.yaml down` |

`Ctrl+C` detiene el montaje que muestra los logs. `down` retira los contenedores del proyecto. No elimina la red host del motor Docker. Conserva imágenes y archivos locales. No necesitas una limpieza global de Docker.

Si cambias código copiado a la imagen, aplica `up --build`. Si sólo cambias variables o la configuración de ejecución, `up` aplica la configuración y recrea cuando corresponde. `restart` reinicia una instancia existente; no reconstruye ni incorpora los cambios de configuración.

## Errores frecuentes

| Síntoma | Qué revisar |
|---|---|
| No encuentra Dockerfile, requisitos o `.env` | Confirma `pwd`: debes estar en `clase-16/`; revisa contexto y nombre del archivo. |
| Host no está disponible o no conecta desde tu computadora | Revisa la versión de Docker Desktop, contenedores Linux y Enable host networking antes de continuar. |
| UI abre pero no predice | Espera a que API termine de cargar; revisa sus logs y que ambos contenedores utilicen host. |
| La UI mantiene una dirección incorrecta | Conserva localhost en `ui/.env` y recrea el contenedor con `--env-file`; respeta override=False. |
| Puerto ocupado o nombre en uso | Identifica tus instancias anteriores con `docker ps -a` y cierra sólo las que pertenecen a tu práctica. |
| Docker advierte que descarta puertos, o Compose rechaza la configuración | Quita -p/ports al utilizar host y conserva network_mode: host en ambos servicios. |
| YAML inválido | Revisa sangría y listas; comprueba con `config --quiet`. |

## Recopilación

La API y Streamlit siguen siendo procesos separados. El modo host comparte la red del anfitrión y permite conservar localhost como destino de API. La configuración entra mediante los `.env`, sin incorporarlos a las imágenes.

Dockerfile describe cómo construir cada imagen y su comando predeterminado. Compose reúne imágenes, nombres, contextos, variables, comandos y modo de red en un archivo. `up --build` construye y ejecuta ese montaje sin repetir los dos `docker run`.

## ✅ Check final de la clase

- [ ] Continué en `clase-16/`, conservando los archivos y nombres existentes.
- [ ] Expliqué por qué fallaba localhost con los contenedores iniciales separados.
- [ ] Comprobé los requisitos de host y ejecuté ambos contenedores con ese modo.
- [ ] Obtuve una predicción desde UI manteniendo la URL local de API.
- [ ] Interpreté contexto, Dockerfile, env_file, command y network_mode en Compose.
- [ ] Entiendo por qué la plantilla host omite ports y una red propia.
- [ ] Ejecuté `up --build` y comprobé la misma predicción con Compose.
- [ ] Cerré sólo los recursos de mi práctica.

## Referencias

- Docker: [modo host y configuración de Docker Desktop](https://docs.docker.com/engine/network/drivers/host/).
- Docker: [introducción a Compose](https://docs.docker.com/compose/intro/compose-application-model/), [build y Dockerfile](https://docs.docker.com/reference/compose-file/build/), [servicios, puertos y command](https://docs.docker.com/reference/compose-file/services/) y [publicación de puertos](https://docs.docker.com/get-started/docker-concepts/running-containers/publishing-ports/).
- Docker: [variables de entorno](https://docs.docker.com/compose/how-tos/environment-variables/set-environment-variables/), [redes Compose](https://docs.docker.com/compose/how-tos/networking/) y [orden de arranque](https://docs.docker.com/compose/how-tos/startup-order/).
- [python-dotenv](https://bbc2.github.io/python-dotenv/).

Código y explicaciones propios bajo MIT. Las referencias externas conservan sus licencias.